# Training-free SmolVLM MVPruner: full test, two GPUs
Enable Internet and **two GPUs**. Attach `huyhuy12334/cache-smolvlm` as Input. Runs zero-based layer indices 2 and 15 independently, each FP32 batch 64 over all 16,817 test records. Parent validates immutable cache once; concurrent longest-prompt stress must succeed before concurrent full evaluation. No OOM retry, batch reduction, mixed precision or training. Java and all four caption metrics are checked before weights load.


In [ ]:
from pathlib import Path
import os, subprocess, sys
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
SOURCE = Path('/kaggle/working/Log_Metrics')
if not SOURCE.exists():
    subprocess.run(['git','clone','--depth','1','--branch','smolvlm-attention',
                    'https://github.com/huyvanzzz/Log_Metrics.git',str(SOURCE)],check=True)
else:
    branch = subprocess.check_output(['git','-C',str(SOURCE),'branch','--show-current'],text=True).strip()
    origin = subprocess.check_output(['git','-C',str(SOURCE),'remote','get-url','origin'],text=True).strip()
    if branch != 'smolvlm-attention' or origin.rstrip('/') != 'https://github.com/huyvanzzz/Log_Metrics.git':
        raise RuntimeError('Source branch/origin mismatch; choose a new source directory')
    if subprocess.check_output(['git','-C',str(SOURCE),'status','--porcelain','--untracked-files=no'],text=True).strip():
        raise RuntimeError('Source has tracked local changes; preserve them before updating')
    subprocess.run(['git','-C',str(SOURCE),'pull','--ff-only','origin','smolvlm-attention'],check=True)
os.chdir(SOURCE)
sys.path.insert(0,str(SOURCE))
subprocess.run([sys.executable,'-m','pip','install','-r',str(SOURCE/'smolvlm_inference/requirements.txt'),
                '-r',str(SOURCE/'smolvlm_inference/requirements-pruning-metrics.txt')],check=True)


In [ ]:
import torch, transformers, tokenizers
assert transformers.__version__ == '4.46.3'
assert tokenizers.__version__ == '0.20.4'
assert torch.cuda.device_count() == 2, 'Enable two Kaggle GPUs'
from smolvlm_inference.pruning_run import preflight_metrics
preflight_metrics()


In [ ]:
from huggingface_hub import hf_hub_download
import kagglehub
OUTPUT = Path('/kaggle/working/smolvlm_mvpruner_test')
OUTPUT.mkdir(exist_ok=True)
PREPARED = OUTPUT.parent/'mvpruner_prepared'
PREPARED_REVISION = 'd2a6914f681dbb9148c4ac11e6fa92452010506e'
CHECKPOINT_REVISION = 'f9d4e9a4d704078f9e7f39f061d13602cf251c78'
for name in ('manifest.json','train.jsonl','val.jsonl','test.jsonl'):
    hf_hub_download('huyvanzzz/prepared_data',repo_type='dataset',revision=PREPARED_REVISION,
                    filename=name,local_dir=PREPARED,token=False)
CHECKPOINT = hf_hub_download('huyvanzzz/smolvlm256M_base',revision=CHECKPOINT_REVISION,
                            filename='finetune_best.pt',token=False)
CACHE_INPUT_DIR = '' # Optional exact mounted cache directory; nothing is copied.
root = Path(CACHE_INPUT_DIR) if CACHE_INPUT_DIR else Path(kagglehub.dataset_download('huyhuy12334/cache-smolvlm'))
candidates = [p.parent for p in root.rglob('features.float16.memmap')
              if (p.parent/'manifest.json').is_file() and (p.parent/'index.json').is_file()]
if len(candidates) != 1:
    raise RuntimeError(f'Expected one complete cache, found {len(candidates)}')
CACHE = candidates[0].resolve()
RECEIPT = OUTPUT/'validation_receipt.json'


In [ ]:
# Parent: one bounded full finiteness pass; no model weights allocated.
from smolvlm_inference.pruning_run import validate_cache, cache_identity
cache = validate_cache(CACHE, PREPARED, cache_identity(), RECEIPT)
assert cache.counts['test'] == 16817
# Workers independently recheck receipt paths/stats, metadata hashes and complete coverage.
# Both stress jobs (normal greedy plus forced 512-step cache stress) finish before test starts.
subprocess.run([sys.executable,'-m','smolvlm_inference.pruning_run','launch',
    '--cache',str(CACHE),'--prepared',str(PREPARED),'--receipt',str(RECEIPT),
    '--checkpoint',str(CHECKPOINT),'--output',str(OUTPUT)],check=True)


In [ ]:
# Metrics run sequentially after both full-test jobs; display source-scaled values.
import json
from IPython.display import FileLink, Markdown, display
scores = {index:json.loads((OUTPUT/f'layer_index_{index}'/'metrics_display.json').read_text()) for index in (2,15)}
lines = ['| Metric | Layer index 2 | Layer index 15 |','|---|---:|---:|']
for name in ('BLEU-1','BLEU-2','BLEU-3','BLEU-4','METEOR','ROUGE-L','CIDEr'):
    lines.append(f"| {name} | {scores[2][name]:.6f} | {scores[15][name]:.6f} |")
display(Markdown('\n'.join(lines)))
display(FileLink(str(OUTPUT/'mvpruner_artifacts.zip')))
